# `tl.som`'s vectorized BMU assignment vs `minisom.MiniSom.winner()`

`tl.som` trains a real `minisom.MiniSom` object (`minisom` is an existing astrolabe
optional extra via `astrolabe[som]`, not a new dependency) but assigns every cell to
its best-matching unit (BMU) with a private, vectorized `_batch_bmu` helper rather
than looping over minisom's own per-sample `.winner()` - batching the squared-
Euclidean argmin over chunks of cells avoids materializing the full distance array
at once. Since `tl.som()` returns the trained `MiniSom` object directly, this
notebook needs no private import: it reruns `.winner()` on the same trained codebook
astrolabe just built, isolating exactly the one thing being validated - does the
vectorized BMU assignment agree with minisom's own reference finder.

In [1]:
import logging
logging.basicConfig(level=logging.INFO)

import time
import numpy as np
import astrolabe as ab


INFO:astrolabe.pl._theme:Registered bundled Inter font files with matplotlib


INFO:astrolabe.pl._theme:astrolabe plot theme set to 'light'


## Dataset

`ab.datasets.xenium_tiny()` with a PCA embedding - `tl.som`'s default
`use_rep='X_pca'` trains directly on this.

In [2]:
mdata = ab.datasets.xenium_tiny()
ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)
ab.pp.highly_variable_genes(mdata)
ab.tl.pca(mdata)

adata = mdata["xenium"]
adata


INFO:astrolabe.io._xenium:Reading Xenium run 'outs' from C:\Users\Austin\PycharmProjects\astrolabe\benchmarks\minisom\.astrolabe_cache\datasets\xenium_tiny\outs


C:\Users\Austin\PycharmProjects\astrolabe\astrolabe\io\_xenium.py:870: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  adata.obsm[obsm_key] = ak.Array(
INFO:astrolabe.io._xenium:Loaded 358 cells × 520 features (115 control probes)


INFO:astrolabe.io._xenium:Streaming transcripts from C:\Users\Austin\PycharmProjects\astrolabe\benchmarks\minisom\.astrolabe_cache\datasets\xenium_tiny\outs\transcripts.parquet


INFO:astrolabe.io._xenium:Loaded 20999 transcript detections


INFO:astrolabe.pipeline:filter_xenium: 358 cells × 520 genes


...

INFO:astrolabe.pp._qc:filter_genes: removing 115 / 370 features


INFO:astrolabe.pp._qc:  synced uns['transcripts']: 20163 -> 20160 rows (dropped filtered-out genes)


INFO:astrolabe.pipeline:  [remove_controls] 370 → 255 genes  (removed 115, 31.1%)


INFO:astrolabe.pipeline:filter_xenium: done — 328 cells × 255 genes


AnnData object with n_obs × n_vars = 328 × 255
    obs: 'cell_area', 'nucleus_area', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'nucleus_count', 'unassigned_codeword_total', 'deprecated_codeword_total', 'genomic_control_probe_total', 'negative_control_codeword_total', 'negative_control_probe_total', 'any_negative_control_probe_total', 'n_unassigned_codewords', 'n_deprecated_codewords', 'n_genomic_control_probes', 'n_negative_control_codewords', 'n_negative_control_probes', 'n_any_negative_control_probes', 'pct_unassigned_codewords', 'pct_deprecated_codewords', 'pct_genomic_control_probes', 'pct_negative_control_codewords', 'pct_negative_control_probes', 'pct_any_negative_control_probes', 'pct_counts_control', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'nucleus_to_cell_area_ratio', 'has_nucleus', 'total_counts_norm'
    var: 

## Astrolabe: `tl.som`

`x=8, y=8` gives a modest 64-node grid, small enough that a full Python
`.winner()` loop below finishes quickly.

In [3]:
_t0 = time.perf_counter()
som_obj = ab.tl.som(mdata, x=8, y=8, seed=42)
t_astro_s = time.perf_counter() - _t0
print(f"astrolabe runtime (train + BMU assignment): {t_astro_s:.4f}s")
bmu_astro = adata.obsm["X_som"]  # (n_cells, 2) [col, row]
bmu_astro[:10]


INFO:astrolabe.tl._som:som: training 8×8 grid on 328 cells (50 features, 32800 iterations, sigma=2.00)


INFO:astrolabe.tl._som:som: done → obsm['X_som'], obs['som_cluster'], uns['som']


astrolabe runtime (train + BMU assignment): 1.0652s


array([[6., 4.],
       [6., 5.],
       [6., 5.],
       [6., 4.],
       [6., 2.],
       [6., 4.],
       [6., 5.],
       [3., 2.],
       [5., 6.],
       [2., 5.]], dtype=float32)

## Reference: `minisom.MiniSom.winner()` looped over the same training array

`som_obj` is the exact trained codebook astrolabe just produced - this loop asks
minisom's own reference BMU finder the same question astrolabe's vectorized code
answered, on the same weights.

In [4]:
data = np.asarray(adata.obsm["X_pca"], dtype=np.float32)
_t0 = time.perf_counter()
bmu_ref = np.array([som_obj.winner(sample) for sample in data], dtype=np.float32)
t_ref_s = time.perf_counter() - _t0
print(f"reference runtime (BMU assignment only, pre-trained codebook): {t_ref_s:.4f}s")
bmu_ref[:10]


reference runtime (BMU assignment only, pre-trained codebook): 0.0060s


array([[6., 4.],
       [6., 5.],
       [6., 5.],
       [6., 4.],
       [6., 2.],
       [6., 4.],
       [6., 5.],
       [3., 2.],
       [5., 6.],
       [2., 5.]], dtype=float32)

## Agreement

BMU coordinates are discrete grid labels, not continuous measurements - Pearson r
is the wrong tool here (two columns of small integers can show a misleadingly high
or low r depending on the grid's aspect ratio). Use exact match rate and Cohen's
kappa instead.

In [5]:
from sklearn.metrics import cohen_kappa_score

exact_match = np.all(bmu_astro == bmu_ref, axis=1)
match_rate = exact_match.mean()

# Cohen's kappa on the flattened "{col}_{row}" node label, treating BMU
# assignment as a categorical prediction task.
labels_astro = [f"{int(c)}_{int(r)}" for c, r in bmu_astro]
labels_ref = [f"{int(c)}_{int(r)}" for c, r in bmu_ref]
kappa = cohen_kappa_score(labels_astro, labels_ref)

print(f"Exact match rate = {match_rate:.4%}, Cohen's kappa = {kappa:.4f}")
print(f"{(~exact_match).sum()} / {len(exact_match)} cells disagree")
print(f"Runtime: astrolabe (train+BMU) {t_astro_s:.4f}s vs minisom (BMU only) {t_ref_s:.4f}s -- "
      f"not apples-to-apples, astrolabe's number includes SOM training the reference call skips.")


Exact match rate = 100.0000%, Cohen's kappa = 1.0000
0 / 328 cells disagree
Runtime: astrolabe (train+BMU) 1.0652s vs minisom (BMU only) 0.0060s -- not apples-to-apples, astrolabe's number includes SOM training the reference call skips.


## Interpretation

Both are an argmin over squared Euclidean distance to the same 64-node codebook,
so a near-100% exact match rate is expected. Remaining disagreement is most likely
floating-point tie-breaking on cells that sit almost exactly equidistant between
two nodes, not an algorithmic difference - check whether disagreeing cells cluster
near node boundaries in `hit_counts`/`umatrix` before suspecting a real bug in
`_batch_bmu`.